In [ ]:
#!pip install langchain langchain-community pypdf2 python-dotenv openai tiktoken

   ---------------------------------------- 0.0/2.5 MB ? eta -:--:--
   -------------------- ------------------- 1.3/2.5 MB 6.4 MB/s eta 0:00:01
   ---------------------------------------- 2.5/2.5 MB 6.2 MB/s eta 0:00:00
   ---------------------------------------- 0.0/879.1 kB ? eta -:--:--
   ---------------------------------------- 879.1/879.1 kB 5.9 MB/s eta 0:00:00
   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------- 1.0/1.0 MB 5.9 MB/s eta 0:00:00
   ---------------------------------------- 0.0/542.4 kB ? eta -:--:--
   ---------------------------------------- 542.4/542.4 kB 5.3 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.3
    Uninstalling requests-2.32.3:
      Successfully uninstalled requests-2.32.3


In [ ]:
#!pip install langchain langchain-text-splitters

Data Sources

PDF — NIST AI Risk Management Framework (AI RMF 1.0)
This is the perfect Trustworthy AI PDF. It defines seven key trustworthiness characteristics: valid and reliable, safe, secure and resilient, accountable and transparent, explainable and interpretable, privacy-enhanced, and fair with harmful bias managed.
Download directly: https://nvlpubs.nist.gov/nistpubs/ai/nist.ai.100-1.pdf
It is free, official, well-structured with headers and sections — perfect for the chunking lab.

Podcast Transcript — "AI Quick Bits" by Scot Pansing
AI Quick Bits is a podcast about artificial intelligence hosted by Scot Pansing. 
This is ideal because it covers the same Trustworthy AI themes as the NIST PDF (safety, ethics, regulation) and transcripts are freely available as text.
https://aiquickbits.com/interview-with-reid-blackman-ceo-of-virtue-and-author-of-ethical-machines

In [7]:
import sys
!{sys.executable} -m pip install langchain langchain-text-splitters -q

In [8]:
# NEW correct import for langchain >= 0.2
from langchain_text_splitters import CharacterTextSplitter
import matplotlib.pyplot as plt
import numpy as np

In [11]:
import sys
!{sys.executable} -m pip install pdfplumber -q
print("✓ pdfplumber installed")

✓ pdfplumber installed


In [14]:
import pdfplumber
import os

pdf_path = "nist.ai.100-1.pdf"

with pdfplumber.open(pdf_path) as pdf:
    pages_text = []
    for page in pdf.pages:
        text = page.extract_text()
        if text:
            pages_text.append(text)
    pdf_text = "\n".join(pages_text)
    total_pages = len(pdf.pages)

with open("nist_ai_rmf.txt", "w", encoding="utf-8") as f:
    f.write(pdf_text)

print(f"✓ Extracted {len(pdf_text):,} characters")
print(f"  Total pages: {total_pages}")
print(f"  Pages with text: {len(pages_text)}")
print(f"  Saved to: nist_ai_rmf.txt")
print(f"\nPreview (first 300 chars):")
print(pdf_text[:300])

✓ Extracted 101,233 characters
  Total pages: 48
  Pages with text: 48
  Saved to: nist_ai_rmf.txt

Preview (first 300 chars):
NIST AI 100-1
Artificial Intelligence Risk Management
Framework (AI RMF 1.0)
NIST AI 100-1
Artificial Intelligence Risk Management
Framework (AI RMF 1.0)
Thispublicationisavailablefreeofchargefrom:
https://doi.org/10.6028/NIST.AI.100-1
January2023
U.S.DepartmentofCommerce
GinaM.Raimondo,Secretary
Na


In [16]:
import shutil
import os

# Copy audio file to working directory
src  = r"C:\Users\dbyst\Downloads\AIQB-ReidBlackman.mp3"
dst  = os.path.join(os.getcwd(), "AIQB-ReidBlackman.mp3")

shutil.copy2(src, dst)
print(f"✓ Copied to: {dst}")
print(f"  Size: {os.path.getsize(dst)/1024/1024:.1f} MB")

✓ Copied to: c:\Users\dbyst\OneDrive\Desktop\Ironhack_labs\Ironhack_Day10\AIQB-ReidBlackman.mp3
  Size: 39.2 MB


In [19]:
import subprocess
import os

src  = "AIQB-ReidBlackman.mp3"
dst  = "AIQB-ReidBlackman-compressed.mp3"

# Compress to 64kbps mono — reduces size by ~50% with minimal quality loss
subprocess.run([
    "ffmpeg", "-y",
    "-i", src,
    "-b:a", "64k",      # 64kbps bitrate
    "-ac", "1",          # mono (halves file size)
    dst
], capture_output=True)

original_mb   = os.path.getsize(src) / 1024 / 1024
compressed_mb = os.path.getsize(dst) / 1024 / 1024

print(f"Original:   {original_mb:.2f} MB")
print(f"Compressed: {compressed_mb:.2f} MB")
print(f"Reduction:  {(1 - compressed_mb/original_mb)*100:.0f}%")

if compressed_mb < 25:
    print("✓ File is now under 25MB limit — ready to transcribe")
else:
    print("✗ Still too large — will need chunking")

Original:   39.25 MB
Compressed: 19.62 MB
Reduction:  50%
✓ File is now under 25MB limit — ready to transcribe


In [20]:
# ── Transcribe compressed file ────────────────────────────────────
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

print("Transcribing with Whisper — please wait...")

with open("AIQB-ReidBlackman-compressed.mp3", "rb") as f:
    transcript = client.audio.transcriptions.create(
        model="whisper-1",
        file=f,
        response_format="text"
    )

# Save as text file for chunking lab
with open("trustworthy_ai_podcast.txt", "w", encoding="utf-8") as f:
    f.write(transcript)

print(f"✓ Transcribed successfully")
print(f"  Characters: {len(transcript):,}")
print(f"  Words:      {len(transcript.split()):,}")
print(f"  Saved to:   trustworthy_ai_podcast.txt")
print(f"\nPreview (first 300 chars):")
print(transcript[:300])

Transcribing with Whisper — please wait...
✓ Transcribed successfully
  Characters: 44,383
  Words:      8,232
  Saved to:   trustworthy_ai_podcast.txt

Preview (first 300 chars):
Hey everyone, welcome back to AI Quick Bits. I'm your host, Scott Panzing, and in this episode I discuss the world of AI ethics and responsible AI, and ethics in general, with Reid Blackman, founder and CEO of Virtue, an AI ethical risk consultancy. Reid is also the author of the book Ethical Machin


Step 2: Fixed-Size Chunking

In [ ]:
Step 2: Implement Fixed-Size Chunking
